# AMOS 预处理（Colab）

与 `colab_ribfrac_preprocess.ipynb`、`colab_tcia_covid19_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` **相同的处理逻辑与超参**：
- 每个 nii volume：**50 个 npy**（z≥128 时）或 **33 个 npy**（z<128 时，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**数据**：仅处理解压后 `amos22/amos22/` 下 **`imagesTr` / `imagesTs` / `imagesVa`** 中的 `.nii.gz`（CT），**不处理** `labels*`；路径含 **`__MACOSX`** 的自动跳过。

**输出**：所有 `.npy` 直接放在与 `unzip/` **平级**的 **`npy/`** 根目录下（不再分子文件夹）；文件名含 split 前缀（如 `imagesTr_amos_0001_0_1.npy`）以防重名。

In [1]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2: 安装依赖
!pip install -q nibabel monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 22.8 MB/s eta 0:00:00


In [3]:
# Cell 3: AMOS 预处理（与 TCIA / RibFrac / proc_spatial_sequence 逻辑一致）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

# ========== 配置 ==========
DRIVE_AMOS_UNZIP = '/content/drive/MyDrive/dataset/pretrain/Amos/unzip'
# nnU-Net 风格根目录（amos22.zip 解压后多为 unzip/amos22/amos22/）
AMOS_DATA_ROOT = os.path.join(DRIVE_AMOS_UNZIP, 'amos22', 'amos22')
# 与 unzip 平级，所有 patch 的 .npy 扁平放在该目录下
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/pretrain/Amos/npy'

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0

ALLOWED_IMAGE_DIRS = frozenset({'imagesTr', 'imagesTs', 'imagesVa'})


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_amos(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-2] if len(path_parts) >= 2 else 'Amos'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_amos_ct_nii_files(data_root):
    """仅 imagesTr / imagesTs / imagesVa；跳过 __MACOSX。"""
    nii_files = []
    if not os.path.isdir(data_root):
        return nii_files
    for dirpath, _, files in os.walk(data_root):
        norm = dirpath.replace('\\', '/')
        if '__MACOSX' in norm:
            continue
        if os.path.basename(dirpath) not in ALLOWED_IMAGE_DIRS:
            continue
        for f in files:
            if f.endswith('.nii.gz') or f.endswith('.nii'):
                nii_files.append(os.path.join(dirpath, f))
    return sorted(nii_files)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_amos_ct_nii_files(AMOS_DATA_ROOT)
print(f'AMOS 数据根: {AMOS_DATA_ROOT}')
print(f'找到 {len(data_list)} 个 CT nii（仅 imagesTr/Ts/Va）')

if len(data_list) == 0:
    print('未找到文件，请检查 AMOS_DATA_ROOT 是否与解压结构一致（amos22/amos22/）。')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='预处理')):
        patch_list = cut_and_save_one_volume_amos(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 20 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}')

AMOS 数据根: /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22
找到 600 个 CT nii（仅 imagesTr/Ts/Va）


预处理:   3%|▎         | 20/600 [03:39<51:59,  5.38s/it]

[20/600] /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22/imagesTr/amos_0033.nii.gz -> 33 npy


预处理:   7%|▋         | 40/600 [05:30<51:01,  5.47s/it]

[40/600] /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22/imagesTr/amos_0067.nii.gz -> 33 npy


...

[560/600] /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22/imagesVa/amos_0328.nii.gz -> 50 npy


预处理:  97%|█████████▋| 580/600 [1:26:45<02:41,  8.09s/it]

[580/600] /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22/imagesVa/amos_0409.nii.gz -> 33 npy


预处理: 100%|██████████| 600/600 [1:29:01<00:00,  8.90s/it]

[600/600] /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22/amos22/imagesVa/amos_0598.nii.gz -> 50 npy

完成! 共 24169 个 npy, 输出: /content/drive/MyDrive/dataset/pretrain/Amos/npy


In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt 已保存: {list_save_dir}')